## 1. Import Requirements

# Master Shear-Wave Splitting Workflow for Axial Seamount

This notebook provides a complete, clean workflow from raw earthquake catalog and waveform data to shear-wave splitting analysis results. The workflow follows proper sequencing and includes all necessary quality control measures. 

Instead of using catalog from Wilcock and Zhang or ML DD, we use the nlloc file for all stations from Christian's results.

## Workflow Overview

1. **Data Loading & Initial Setup** - Load earthquake catalog and station metadata
2. **Extended Time Window Creation** - Create proper time windows for waveform retrieval
3. **Waveform Data Retrieval** - Download seismic data with extended windows
4. **Quality Control Filters** - P-wave rectilinearity, SNR, and incidence angle filtering
5. **Geometric Calculations** - Back-azimuth and distance calculations
6. **Shear-Wave Splitting Analysis** - Dynamic parameter estimation and SWSPy analysis
7. **Results Processing & Visualization** - Compile and visualize splitting parameters

## Key Improvements
- Extended catalog creation moved to proper early position
- Updated P-wave polarization analysis for true incidence angles
- Integrated SNR calculations with proper S-wave timing
- Clean separation of quality control steps

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Load 2018 earthquake test catalog - ML DD
#catalog = pd.read_csv('2018_eq_catalog.csv')

# Load Baillard nonlinloc catalog
catalog = pd.read_csv('AXIAL.PHASE.FINAL_3D_V2.csv')

#Load station information from Christian's data
stations_file = '../data/stations_axial.llz'
#Read llz file - reads like a text file with space delimiter
stations_df = pd.read_csv(stations_file, delim_whitespace=True, header=None, names=['Longitude (°W)', 'Latitude (°N)', 'Elevation (m)', 'Station ID'])
# Convert elevation column to m from km
stations_df['Elevation (m)'] = stations_df['Elevation (m)']*1000

print(f"Stations in catalog: {catalog['station'].value_counts()}")

In [ ]:
# Filter catalog for AXAS2 station only
#axas2_catalog = catalog[catalog['station'] == 'OOAXAS2'].copy()

#axas2_catalog = catalog[catalog['station'] == 'AXAS2'].copy()

# Look at AXEC2 station
axec2_catalog = catalog[catalog['station'] == 'AXEC3'].copy()

# Reset index to ensure clean indexing
#axas2_catalog = axas2_catalog.reset_index(drop=True)
axec2_catalog = axec2_catalog.reset_index(drop=True)

# Pick all events from April and May, 2015
#axas2_catalog['datetime'] = pd.to_datetime(axas2_catalog['datetime'])
#axas2_catalog = axas2_catalog[(axas2_catalog['datetime'] >= '2015-04-20') & (axas2_catalog['datetime'] < '2015-04-28')].copy()

axec2_catalog['datetime'] = pd.to_datetime(axec2_catalog['datetime'])
axec2_catalog = axec2_catalog[(axec2_catalog['datetime'] >= '2015-04-20') & (axec2_catalog['datetime'] < '2015-04-28')].copy()

# Select first 100 events for testing
#test_catalog_100 = axas2_catalog.head(100).copy()


#print(f"Total AXAS2 events in catalog: {len(axas2_catalog)}")
print(f"Total AXEC2 events in catalog: {len(axec2_catalog)}")

#print(f"Test catalog created with first {len(test_catalog_100)} AXAS2 events")
print(f"\nDate range of test catalog:")
print(f"Start: {axec2_catalog['datetime'].min()}")
print(f"End: {axec2_catalog['datetime'].max()}")

display(axec2_catalog)

In [ ]:
# Downsample catalog by a factor of 4 - take every fourth event
#test_catalog = axas2_catalog.iloc[::4].copy()

test_catalog = axec2_catalog.copy()

print(f"\nDate range of test catalog:")
print(f"Start: {test_catalog['datetime'].min()}")
print(f"End: {test_catalog['datetime'].max()}")

In [ ]:
# First, reformat the datetime strings to add 'T' separator
test_catalog['p_time'] = test_catalog['p_time'].str.replace(' ', 'T', regex=False)
test_catalog['s_time'] = test_catalog['s_time'].str.replace(' ', 'T', regex=False)
test_catalog['datetime'] = test_catalog['datetime'].astype(str).str.replace(' ', 'T', regex=False)

# Now convert to pandas Timestamp with UTC timezone
test_catalog['p_time'] = pd.to_datetime(test_catalog['p_time'], utc=True, format='ISO8601')
test_catalog['s_time'] = pd.to_datetime(test_catalog['s_time'], utc=True, format='ISO8601')
test_catalog['datetime'] = pd.to_datetime(test_catalog['datetime'], utc=True, format='ISO8601')

# Convert to UTCDateTime
test_catalog['p_time'] = test_catalog['p_time'].apply(lambda x: UTCDateTime(x))
test_catalog['s_time'] = test_catalog['s_time'].apply(lambda x: UTCDateTime(x))
test_catalog['datetime'] = test_catalog['datetime'].apply(lambda x: UTCDateTime(x))

print("Successfully converted to UTCDateTime")
print(f"Sample p_time: {test_catalog['p_time'].iloc[0]}")

In [ ]:
catalog = test_catalog.copy()

## 3. Extended Time Window Creation

This step creates extended time windows for waveform retrieval. This is critical for proper analysis and must happen early in the workflow, before any quality control that depends on waveform data.

In [ ]:
# Create extended time windows for proper waveform analysis
print("Creating extended time windows for waveform retrieval...")

# Apply extended windowing
extended_catalog = create_extended_catalog(catalog, pre_p_time=1.0, post_s_time=2.0)

print(f"Extended catalog created with {len(extended_catalog)} events")
print(f"Time windows: {extended_catalog['total_duration'].iloc[0]} seconds total")
print(f"Pre-event: {extended_catalog['pre_p_sec'].iloc[0]}s, Post-event: {extended_catalog['post_s_sec'].iloc[0]}s")
# Display sample of extended timing
print("\nSample timing windows:")
sample_cols = ['id', 'datetime', 'starttime', 'endtime', 'total_duration']
display(extended_catalog[sample_cols].head())

In [ ]:
# Remove leading 'OO' from station names
extended_catalog['station'] = extended_catalog['station'].str.replace('OO', '', regex=False)

In [ ]:
display(extended_catalog)

## 4. Waveform Data Retrieval

This section retrieves seismic waveform data using the extended time windows. We'll load the existing trace data and organize it for processing.

In [ ]:
test_catalog = extended_catalog

In [ ]:
# Replace catalog id with index
test_catalog['id'] = test_catalog.index

test_catalog['mag'] = 0.0

In [ ]:
def get_station_traces_batch(df, filename, starttime, endtime, station_id, batch_size=250):
    """
    Fast bulk retrieval of waveform data with intelligent batched fallback.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with event information
    filename : str
        Output filename (without extension)
    starttime : str
        Column name for start time
    endtime : str
        Column name for end time
    station_id : str
        Column name for station ID
    
    Returns:
    --------
    obspy.Stream : All retrieved traces
    
    Performance:
    - Bulk success: ~10-15 seconds for 250 events
    - Batched fallback: ~30-60 seconds (250 events per batch)
    - Individual fallback: Only for failed batches
    """
    from obspy.clients.fdsn import Client
    from obspy.core.utcdatetime import UTCDateTime
    from obspy import Stream
    import time
    
    client = Client("IRIS")
    all_traces = Stream()

    # BATCHED- 250 events per batch
    batch_size = batch_size
    total_batches = (len(df) + batch_size - 1) // batch_size
    
    for batch_idx in range(total_batches):
        batch_start_idx = batch_idx * batch_size
        batch_end_idx = min(batch_start_idx + batch_size, len(df))
        batch_df = df.iloc[batch_start_idx:batch_end_idx]
        
        print(f"\n{'─'*60}")
        print(f"BATCH {batch_idx + 1}/{total_batches}")
        print(f"Events {batch_start_idx + 1} to {batch_end_idx} ({len(batch_df)} events)")
        print(f"{'─'*60}")
        
        # Build bulk request for this batch
        batch_bulk_list = []
        for _, row in batch_df.iterrows():
            t_start = UTCDateTime(row[str(starttime)]) - 0.5
            t_final = UTCDateTime(row[str(endtime)]) + 0.5
            current_station = row[str(station_id)]
            
            if current_station == 'AXEC2':
                batch_bulk_list.append(('OO', 'AXEC2', '', 'HHE', t_start, t_final))
                batch_bulk_list.append(('OO', 'AXEC2', '', 'HHN', t_start, t_final))
                batch_bulk_list.append(('OO', 'AXEC2', '', 'HHZ', t_start, t_final))
        
        # Try batch bulk request
        batch_start_time = time.time()
        batch_stream = client.get_waveforms_bulk(batch_bulk_list)
        batch_elapsed = time.time() - batch_start_time
        
        all_traces += batch_stream
        
        print(f"✓ Batch {batch_idx + 1} SUCCESS: {len(batch_stream)} traces in {batch_elapsed:.1f}s")
        print(f"  Expected: {len(batch_bulk_list)}, Retrieved: {len(batch_stream)}")
        
        if len(batch_stream) < len(batch_bulk_list):
            missing = len(batch_bulk_list) - len(batch_stream)
            print(f"  ⚠ Warning: {missing} traces missing from this batch")
            

            
            print(f"\n{'='*60}")
            print(f"BATCHED FALLBACK COMPLETE")
            print(f"{'='*60}")
            print(f"Total traces retrieved: {len(all_traces)}")
    
    # Save results
    if all_traces:
        print(f"\nSaving {len(all_traces)} traces to {filename}.mseed...")
        all_traces.write(str(filename) + ".mseed", format="MSEED")
        print(f"✓ File saved successfully")
        
        # Summary statistics
        print(f"\n{'='*60}")
        print(f"RETRIEVAL SUMMARY")
        print(f"{'='*60}")
        print(f"Total events processed: {len(df)}")
        print(f"Total traces retrieved: {len(all_traces)}")
        print(f"Expected traces (max): {len(df) * 3}")
        print(f"Success rate: {len(all_traces)/(len(df)*3)*100:.1f}%")
        
    else:
        print(f"\n⚠ WARNING: No traces retrieved!")
    
    return all_traces

In [ ]:
# Retrieve waveforms for all events in the test catalog using get_all_traces function
print("Retrieving waveforms for all events in the test catalog...")
waveforms = get_station_traces_batch(test_catalog, 'axial_nonlinloc_april_20_28_axec2', 'starttime', 'endtime', 'station', batch_size=100)

In [ ]:
# Load waveforms from mseed file with obspy
waveforms_file = 'axial_nonlinloc_april_20_28_axec2.mseed'
waveforms = obspy.read(waveforms_file)

In [ ]:
# Associate waveforms with events in the catalog
print("Organizing waveforms by events...")
waveform_dict = organize_stream_by_events(waveforms, test_catalog)

In [ ]:
# Organize waveforms by event ID
print("Organizing waveforms by event ID...")
organized_waveforms = organize_waveform_data(waveform_dict, test_catalog)

In [ ]:
# Format s_arrival_time and p_arrival_time as difference between arrival times and origin time
print("Formatting s_arrival_time and p_arrival_time as differences from origin time...")
for eid in organized_waveforms.keys():
    organized_waveforms[eid]['s_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 's_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))
    organized_waveforms[eid]['p_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'p_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))

In [ ]:
# For all traces in organized_waveforms, taper and filter in-place
print("Tapering and filtering all traces in organized_waveforms...")
events_to_remove = []
try:
    for eid in organized_waveforms.keys():
        for tr in organized_waveforms[eid]['traces']:
            tr.detrend("linear") # to avoid weird start and end amplitudes
            tr.taper(max_percentage=0.05, type='hann')
            tr.filter('bandpass', freqmin=5.0, freqmax=40.0)
except Exception as e:
    print(f"Error during waveform processing: {e}")
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
    print(f"Events with issues: {events_to_remove}")

print("Waveform retrieval and organization complete.")

In [ ]:
# Find streams that are NoneType and remove from organized_waveforms
print("Checking for NoneType streams in organized_waveforms...")
events_to_remove = []
for eid in organized_waveforms.keys():
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
if events_to_remove:
    print(f"Removing {len(events_to_remove)} events with NoneType streams: {events_to_remove}")
    for eid in events_to_remove:
        del organized_waveforms[eid]
else:
    print("No NoneType streams found in organized_waveforms.")

In [ ]:
# Remove duplicate traces from organized_waveforms
print("Checking for and removing duplicate traces in organized_waveforms...")

for eid in organized_waveforms.keys():
    # Get the stream for this event
    st = organized_waveforms[eid]['traces']
    
    # Check if there are duplicates
    try:
        if type(st) == type(None):
            print(f"Event {eid}: No traces found (NoneType)")
            continue

        else:
            if len(st) > 3:
                print(f"Event {eid}: Found {len(st)} traces (expected 3)")
                
                # Create a new stream with unique traces based on channel code
                unique_traces = {}
                for tr in st:
                    channel = tr.stats.channel
                    # Keep the first occurrence of each channel
                    if channel not in unique_traces:
                        unique_traces[channel] = tr
                
                # Replace the stream with deduplicated traces
                organized_waveforms[eid]['traces'] = obspy.Stream(traces=list(unique_traces.values()))
                print(f"  Reduced to {len(organized_waveforms[eid]['traces'])} unique traces")
    except Exception as e:
        print(f"Error processing event {eid}: {e}")
        organized_waveforms[eid]['traces'] = st[:3]  # Fallback to first 3 traces if error occurs

# Verify the results
print("\nVerification of trace counts after deduplication:")
trace_counts = {}
for eid in organized_waveforms.keys():
    count = len(organized_waveforms[eid]['traces'])
    trace_counts[count] = trace_counts.get(count, 0) + 1

print(f"Events with 3 traces: {trace_counts.get(3, 0)}")
if any(k != 3 for k in trace_counts.keys()):
    print("Events with unexpected trace counts:")
    for count, num_events in trace_counts.items():
        if count != 3:
            print(f"  {num_events} events with {count} traces")
else:
    print("All events have exactly 3 traces (E, N, Z)")

In [ ]:
# Remove events that do not have exactly 3 traces
print("\nRemoving events that do not have exactly 3 traces...")
events_to_remove = []
for eid in organized_waveforms.keys():
    if len(organized_waveforms[eid]['traces']) != 3:
        events_to_remove.append(eid)

    # also remove events with any trace that has zero length (indicating a retrieval issue) or empty traces
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

for eid in events_to_remove:
    del organized_waveforms[eid]

In [ ]:
len(organized_waveforms)

## 5. Quality Control Pipeline

This section implements comprehensive quality control measures including P-wave rectilinearity analysis, signal-to-noise ratio calculations, and incidence angle filtering.

In [ ]:
# Define quality control thresholds
QC_THRESHOLDS = {
    'min_snr': 2.0,           # Minimum S-wave signal-to-noise ratio
    'min_rectilinearity': 0.7, # Minimum P-wave rectilinearity
    'max_incidence': 30.0,     # Maximum incidence angle (degrees)
    'min_magnitude': 0.0,      # Minimum event magnitude
}

print("Quality control functions loaded successfully")
print(f"QC Thresholds: {QC_THRESHOLDS}")

In [ ]:
# Check that all traces for same event have same length, and remove events that do not meet this criterion
print("Checking that all traces for the same event have the same length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    trace_lengths = [tr.stats.npts for tr in organized_waveforms[eid]['traces']]
    if len(set(trace_lengths)) != 1:
        print(f"Event {eid} has traces of different lengths: {trace_lengths}, marking for removal")
        events_to_remove.append(eid)

In [ ]:
# Check if any traces are length zero, and if so mark those events for removal
print("Checking for traces with zero length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

In [ ]:
if events_to_remove:
    print(f"Removing {len(events_to_remove)} events that do not have traces of the same length or have zero-length traces: {events_to_remove}")
    for eid in events_to_remove:
        del organized_waveforms[eid]
else:
    print("All events have traces of the same length and no zero-length traces found.")

In [ ]:
# Calculate quality control metrics for organized waveforms
print("Calculating quality control metrics for organized waveforms...")

# 1. Calculate S-wave SNR
organized_waveforms = calculate_snr_for_organized_waveforms(organized_waveforms)

# 2. Calculate geographic back-azimuth, for coordinate rotation later
organized_waveforms = calculate_back_azimuth_for_organized_waveforms(organized_waveforms, stations_df)

# 3. Calculate incidence angle
organized_waveforms = calculate_incidence_angle_eigenvalue_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

#4. Calculate P-wave rectilinearity
organized_waveforms = calculate_rectilinearity_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

In [ ]:
# Create a dataframe with quality control metrics for the test catalog events
qc_metrics_df = pd.DataFrame({
    'event_id': list(organized_waveforms.keys()),
    's_time': [organized_waveforms[eid]['s_arrival_time'] for eid in organized_waveforms.keys()],
    'station': [organized_waveforms[eid]['station'] for eid in organized_waveforms.keys()],
    'back_azimuth': [organized_waveforms[eid]['back_azimuth'] for eid in organized_waveforms.keys()],
    'snr_horizontal': [organized_waveforms[eid]['snr_horizontal'] for eid in organized_waveforms.keys()],
    'incidence': [organized_waveforms[eid]['incidence_eigenvalue_jurkevics'] for eid in organized_waveforms.keys()],
    'rectilinearity': [organized_waveforms[eid]['rectilinearity_jurkevics'] for eid in organized_waveforms.keys()],
    'latitude': [organized_waveforms[eid]['latitude'] for eid in organized_waveforms.keys()],
    'longitude': [organized_waveforms[eid]['longitude'] for eid in organized_waveforms.keys()],
    'depth': [organized_waveforms[eid]['depth'] for eid in organized_waveforms.keys()],
    'origin_time' : [organized_waveforms[eid]['origin_time'] for eid in organized_waveforms.keys()]
})

print(f"Quality control metrics dataframe created with {len(qc_metrics_df)} events")
display(qc_metrics_df)

In [ ]:
qc_metrics_df['s_time'] = qc_metrics_df['origin_time'] + qc_metrics_df['s_time']

In [ ]:
display(qc_metrics_df)

In [ ]:
# Define passing_waveforms as those that meet all QC thresholds
passing_waveforms = apply_quality_control(organized_waveforms, QC_THRESHOLDS)

In [ ]:
# Save the passing_waveforms
metadata_df = save_passing_waveforms(passing_waveforms, output_dir='passing_waveforms_data')
display(metadata_df.head())

In [ ]:
# Reload the data
passing_waveforms_reloaded = load_passing_waveforms(output_dir='passing_waveforms_data')

# Verify the reload worked correctly
print(f"Reloaded events: {len(passing_waveforms_reloaded)}")
print(f"\nSample reloaded event (ID: {list(passing_waveforms_reloaded.keys())[0]}):")
sample_event = passing_waveforms_reloaded[list(passing_waveforms_reloaded.keys())[0]]
print(f"  Station: {sample_event['station']}")
print(f"  Origin time: {sample_event['origin_time']}")
print(f"  Number of traces: {len(sample_event['traces'])}")
print(f"  Back azimuth: {sample_event['back_azimuth']:.2f}°")

## 6. Shear-Wave Splitting Analysis

This section implements the core shear-wave splitting analysis using SWSPy with dynamic parameter estimation and comprehensive quality assessment.

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
#print("Performing shear-wave splitting analysis using SWSPy method...")
#results_baillard = perform_splitting_on_organized_waveforms(passing_waveforms, use_dynamic_params=False, mode='baillard', plot_results=False)

In [ ]:
# Save results_baillard dictionary to CSV file for later analysis
#results_df = pd.DataFrame.from_dict(results_baillard, orient='index')
#results_df.to_csv('results_baillard_nonlinloc_april_20_28.csv')

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy_002_tmid_1_9_2_1 = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=0.02 / 0.0509, last_window_start=0, 
                                                         first_window_end=1.9, last_window_end=2.1, n_win=10, s_pick_uncertainty=0.0509, mode='swspy', 
                                                         plot_results=False)

In [ ]:
if 'results_swspy_002_tmid_1_9_2_1' in locals() and results_swspy_002_tmid_1_9_2_1:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_swspy_002_tmid_1_9_2_1,
        title=f"Fast Direction Rose Plot - Station AXEC2, SWSPy: 0.02s to S-pick, 1.9-2.1 Tmid",
        nbins=36,  # 10° binsd
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

In [ ]:
save_results_csv(results_swspy_002_tmid_1_9_2_1, file_name='splitting_results_axec2_apr_20_28_002_tmid_1_9_2_1')

In [ ]:
# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_swspy_002_tmid_1_9_2_1,
    qc_metrics_df,
    station='AXEC2, SWSPy: 0.02s to S-pick, 1.9-2.1 Tmid',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=1,   # 2 samples
    sigma=1.5
)
plt.show()

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using Baillard method...")
results_baillard = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=2, last_window_start=0, 
                                                         first_window_end=1.5, last_window_end=2.5, n_win=10, s_pick_uncertainty=0.0509, mode='baillard', 
                                                         plot_results=False)

save_results_csv(results_baillard, file_name='splitting_results_baillard_axec2_apr_20_28')

In [ ]:
if 'results_baillard' in locals() and results_baillard:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_baillard,
        title=f"Fast Direction Rose Plot - Station AXEC2, Baillard",
        nbins=36,  # 10° binsd
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_baillard,
    qc_metrics_df,
    station='AXEC2, Baillard',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=1,   # 2 samples
    sigma=1.2
)
plt.show()

In [ ]:
# Plot the rose plot data before the eruption and afterwards - updated plotting function

def plot_fast_direction_rose_eruption_comparison(results_dict, eruption_time=None, 
                                                  title_prefix="Fast Direction Distribution",
                                                  nbins=36, figsize=(16, 7), color='steelblue',
                                                  edgecolor='black', linewidth=0.5):
    """
    Create side-by-side 360° rose plots comparing fast directions before and after eruption.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    eruption_time : UTCDateTime
        Time of eruption onset (default: 2015-04-24T06:00:00)
    title_prefix : str
        Prefix for plot titles
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height) for combined plot
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    if eruption_time is None:
        eruption_time = UTCDateTime(2015, 4, 24, 6)  # Nooner and Chadwick 2016
    
    # Split results into before and after eruption
    results_before = {}
    results_after = {}
    
    for event_id, result in results_dict.items():
        event_time = UTCDateTime(result['result']['event_datetime'])
        if event_time < eruption_time:
            results_before[event_id] = result
        else:
            results_after[event_id] = result
    
    # Create figure with two subplots
    fig = plt.figure(figsize=figsize)
    
    # Before eruption plot (left)
    ax1 = fig.add_subplot(121, projection='polar')
    plot_rose_subplot(results_before, ax1, f"{title_prefix}\nBefore Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    # After eruption plot (right)
    ax2 = fig.add_subplot(122, projection='polar')
    plot_rose_subplot(results_after, ax2, f"{title_prefix}\nAfter Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    plt.tight_layout()
    return fig, (ax1, ax2), (results_before, results_after)


def plot_rose_subplot(results_dict, ax, title, nbins, color, edgecolor, linewidth):
    """
    Helper function to plot rose diagram on a given axis.
    """
    # Extract fast directions (phi) from results
    fast_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    if len(fast_directions) == 0:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, 
                ha='center', va='center', fontsize=14)
        return
    
    fast_directions = np.array(fast_directions)
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean
    original_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']
        original_directions.append(np.deg2rad(phi))
    original_directions = np.array(original_directions)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)


def plot_fast_direction_rose(results_dict, title="Fast Direction Distribution", 
                              nbins=36, figsize=(8, 8), color='steelblue',
                              edgecolor='black', linewidth=0.5):
    """
    Create a 360° polar rose plot (histogram) of fast directions with 180° symmetry.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    title : str
        Title for the plot
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height)
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    # Extract fast directions (phi) from results
    fast_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        # Map -90 to 90 range to 0 to 180, then add symmetric values
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    fast_directions = np.array(fast_directions)
    
    # Create polar histogram
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='polar')
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit (counts are doubled due to symmetry)
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean for original ±90° range
    original_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']
        original_directions.append(np.deg2rad(phi))
    original_directions = np.array(original_directions)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    return fig, ax

In [ ]:
if 'results_swspy_002_tmid_1_9_2_1' in locals() and results_swspy_002_tmid_1_9_2_1 is not None:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose_eruption_comparison(
        results_swspy_002_tmid_1_9_2_1,
        title_prefix=f"Fast Direction Distribution - AXEC2, SWSPy",
        nbins=36,  # 10° bins
        color='steelblue',
        figsize=(16, 7)
    )

In [ ]:
if 'results_baillard' in locals() and results_baillard is not None:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose_eruption_comparison(
        results_baillard,
        title_prefix=f"Fast Direction Distribution - AXEC2, Baillard",
        nbins=36,  # 10° bins
        color='steelblue',
        figsize=(16, 7)
    )
    plt.show()

In [ ]:
def plot_splitting_timeseries_smooth(results_dict, qc_metrics_df, station='AXAS2', 
                                     figsize=(14, 8), x_width_days=5, x_overlap=0.95,
                                     y_width_phi=5, y_width_dt=2, y_overlap=0.95,
                                     sigma=2.0, sampling_rate=200.0):
    """
    Create smoothed 2D histogram time-series plots inspired by Baillard's approach.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    qc_metrics_df : pd.DataFrame
        DataFrame with event metadata including origin_time
    station : str
        Station name for title
    figsize : tuple
        Figure size (width, height)
    x_width_days : float
        Width of moving time window in days
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width_phi : float
        Bin width for phi in radians
    y_width_dt : int
        Bin width for dt in samples (1 sample = 1/sampling_rate seconds)
    y_overlap : float
        Overlap for smoothing in y-direction
    sigma : float
        Gaussian smoothing parameter
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    """
    import matplotlib.gridspec as gridspec
    from matplotlib.dates import DateFormatter
    import matplotlib.dates as mdates
    from scipy.ndimage import gaussian_filter
    
    # Extract data from results
    data_list = []
    for event_id, result in results_dict.items():
        if event_id in qc_metrics_df['event_id'].values:
            origin_time = qc_metrics_df.loc[qc_metrics_df['event_id'] == event_id, 'origin_time'].values[0]
            phi = result['result']['phi']
            dt = result['result']['dt']
            
            # Convert phi from degrees to radians and normalize to -pi/2 to +pi/2
            phi_rad = np.deg2rad(phi)
            phi_rad_norm = ((phi_rad + np.pi/2) % np.pi) - np.pi/2
            
            # Convert dt from seconds to samples
            dt_samples = dt * sampling_rate
            
            data_list.append({
                'time': pd.to_datetime(str(origin_time)),
                'phi_rad': phi_rad_norm,
                'phi_deg': phi,
                'dt_samples': dt_samples,
                'dt_seconds': dt
            })
    
    df = pd.DataFrame(data_list).sort_values('time')
    
    if len(df) == 0:
        print("No data to plot")
        return
    
    # Create figure with gridspec layout
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 1, hspace=0.3, top=0.9, bottom=0.1, 
                          left=0.08, right=0.95)
    
    ax_phi = plt.subplot(gs[0])
    ax_dt = plt.subplot(gs[1], sharex=ax_phi)
    
    # Convert datetime to matplotlib date numbers
    time_nums = mdates.date2num(df['time'])
    
    # === PHI PLOT (in radians) ===
    # Create bins
    time_range = (time_nums.min(), time_nums.max())
    n_time_bins = int((time_range[1] - time_range[0]) / (x_width_days * (1 - x_overlap)))
    n_time_bins = max(20, min(n_time_bins, 100))  # Reasonable limits
    
    # Phi bins from -pi/2 to +pi/2 radians (-1.57 to +1.57)
    phi_bins = np.arange(-np.pi/2, np.pi/2 + y_width_phi*np.pi/180, y_width_phi*np.pi/180)
    n_phi_bins = len(phi_bins) - 1
    
    # Create 2D histogram for phi
    H_phi, xedges_phi, yedges_phi = np.histogram2d(
        time_nums, df['phi_rad'], 
        bins=[n_time_bins, phi_bins],
        range=[time_range, None]
    )
    
    # Normalize by column (each time bin) - "norm_y=True" in Baillard's code
    H_phi_norm = H_phi.copy()
    for i in range(H_phi.shape[0]):
        col_sum = H_phi[i, :].sum()
        if col_sum > 0:
            H_phi_norm[i, :] = H_phi[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_phi_smooth = gaussian_filter(H_phi_norm, sigma=sigma)
    
    # Mask zeros
    H_phi_smooth = np.ma.masked_where(H_phi_smooth < 0.001, H_phi_smooth)
    
    # Plot with imshow for smooth appearance
    extent_phi = [xedges_phi[0], xedges_phi[-1], yedges_phi[0], yedges_phi[-1]]
    im_phi = ax_phi.imshow(H_phi_smooth.T, 
                           origin='lower',
                           aspect='auto',
                           extent=extent_phi,
                           cmap='magma',
                           interpolation='bilinear',  # Smooth interpolation
                           alpha=0.9)
    
    # Colorbar
    cbar_phi = plt.colorbar(im_phi, ax=ax_phi, pad=0.01)
    cbar_phi.set_label('Normalized Density', fontsize=10)
    
    # Format phi axis with radians
    ax_phi.set_ylabel('Fast Direction φ (rad)', fontsize=12, fontweight='bold')
    ax_phi.set_ylim(-np.pi/2, np.pi/2)
    ax_phi.axhline(0, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    ax_phi.grid(True, alpha=0.3, linestyle='--', color='white')

      # Add axvline dashed white line at time = 2015-04-24 05:00:00
    event_time_line = pd.to_datetime('2015-04-24 05:00:00')
    ax_phi.axvline(mdates.date2num(event_time_line), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Set y-ticks in radians
    phi_ticks_rad = np.array([-np.pi/2, -np.pi/3, -np.pi/6, 0, np.pi/6, np.pi/3, np.pi/2])
    ax_phi.set_yticks(phi_ticks_rad)
    
    # Add moving average
    #window_size = max(5, len(df) // 10)
    #if len(df) >= window_size:
    #    df['phi_rad_ma'] = df['phi_rad'].rolling(window=window_size, center=True).mean()
    #    ax_phi.plot(df['time'], df['phi_rad_ma'], 'w-', linewidth=2.5, alpha=0.9)
    #    ax_phi.plot(df['time'], df['phi_rad_ma'], 'black', linewidth=2,
    #               label=f'{window_size}-event moving avg', alpha=0.8)
    #    ax_phi.legend(loc='upper right', fontsize=9, facecolor='white', 
    #                 edgecolor='white', framealpha=0.7)
    
    # === DT PLOT (in samples) ===
    dt_max_samples = np.ma.max(df['dt_samples'])
    # Create bins in samples
    dt_bins = np.arange(0, dt_max_samples + y_width_dt, y_width_dt)  # 40 samples = 0.2 sec at 200 Hz
    n_dt_bins = len(dt_bins) - 1
    
    # Create 2D histogram for dt
    H_dt, xedges_dt, yedges_dt = np.histogram2d(
        time_nums, df['dt_samples'],
        bins=[n_time_bins, dt_bins],
        range=[time_range, None]
    )
    
    # Normalize by column
    H_dt_norm = H_dt.copy()
    for i in range(H_dt.shape[0]):
        col_sum = H_dt[i, :].sum()
        if col_sum > 0:
            H_dt_norm[i, :] = H_dt[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_dt_smooth = gaussian_filter(H_dt_norm, sigma=sigma)
    
    # Mask zeros
    H_dt_smooth = np.ma.masked_where(H_dt_smooth < 0.001, H_dt_smooth)
    
    # Plot with imshow
    extent_dt = [xedges_dt[0], xedges_dt[-1], yedges_dt[0], yedges_dt[-1]]
    im_dt = ax_dt.imshow(H_dt_smooth.T,
                         origin='lower',
                         aspect='auto',
                         extent=extent_dt,
                         cmap='magma',
                         interpolation='bilinear',
                         alpha=0.9)
    
    # Colorbar
    cbar_dt = plt.colorbar(im_dt, ax=ax_dt, pad=0.01)
    cbar_dt.set_label('Normalized Density', fontsize=10)

    # Add axvline dashed white line at time = 2015-04-24 05:00:00
    event_time_line = pd.to_datetime('2015-04-24 05:00:00')
    ax_dt.axvline(mdates.date2num(event_time_line), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Format dt axis with samples
    ax_dt.set_ylabel('Delay Time δt (samples)', fontsize=12, fontweight='bold')
    ax_dt.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax_dt.set_ylim(0, 25)  # Show up to 25 samples (0.125 sec at 200 Hz)
    ax_dt.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Add secondary y-axis for seconds
    #ax_dt_sec = ax_dt.secondary_yaxis('right', functions=(
    #    lambda x: x / sampling_rate,  # samples to seconds
    #    lambda x: x * sampling_rate   # seconds to samples
    #))
    #ax_dt_sec.set_ylabel('δt (s)', fontsize=10)
    
    # Add moving average for dt
    #if len(df) >= window_size:
    #    df['dt_samples_ma'] = df['dt_samples'].rolling(window=window_size, center=True).mean()
    #    ax_dt.plot(df['time'], df['dt_samples_ma'], 'w-', linewidth=2.5, alpha=0.9)
    #    ax_dt.plot(df['time'], df['dt_samples_ma'], 'black', linewidth=2,
    #              label=f'{window_size}-event moving avg', alpha=0.8)
    #    ax_dt.legend(loc='upper right', fontsize=9, facecolor='white',
    #                 edgecolor='white', framealpha=0.7)
    
    # Format x-axis with dates
    date_formatter = DateFormatter('%Y-%m-%d')
    ax_dt.xaxis.set_major_formatter(date_formatter)
    
    # Auto-adjust date locator
    days_span = (df['time'].max() - df['time'].min()).days
    if days_span > 60:
        ax_dt.xaxis.set_major_locator(mdates.MonthLocator())
    elif days_span > 14:
        ax_dt.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    else:
        ax_dt.xaxis.set_major_locator(mdates.DayLocator())
    
    plt.setp(ax_dt.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.setp(ax_phi.xaxis.get_majorticklabels(), visible=False)
    
    # Add title with statistics (in both degrees and radians)
    phi_mean_deg = df['phi_deg'].mean()
    phi_std_deg = df['phi_deg'].std()
    dt_mean_sec = df['dt_seconds'].mean()
    dt_std_sec = df['dt_seconds'].std()
    
    title = (f"Splitting Parameter Time Series - Station {station}\n"
             f"N = {len(df)} events | "
             f"φ: {phi_mean_deg:.1f}° ± {phi_std_deg:.1f}° "
             f"({np.deg2rad(phi_mean_deg):.2f} ± {np.deg2rad(phi_std_deg):.2f} rad) | "
             f"δt: {dt_mean_sec:.3f} ± {dt_std_sec:.3f} s "
             f"({dt_mean_sec*sampling_rate:.1f} ± {dt_std_sec*sampling_rate:.1f} samples)")
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    return fig, (ax_phi, ax_dt), df

In [ ]:
def plot_movehisto2d_time(self,y_param,minlambda_select='min', 
                              x_label='X',y_label='Y',title='',ax=None,
                              angle_mode='trigo',lag_mode='sample',sampling_rate=200,
                              window_unit=None,window_bottom=None,window_top=None,
                              **movehisto2d_kwargs):
        """
        Function made to use movehist_2d and show variations of splitting parameters with time
        parameters are similat to the one for movehisto2d
        Plots showing angles used trigo convention by default (ie CCW from East) however angles can
        be shown using azimuth conention by specifyin angle_mode='azimuth', in that case all paramters
        should be given using that convention (ie. y_width, y_start,y_cycle...)
        
        parameter should be given in degrees
        
        Input
        ----
            angle_mode: str: ['trigo','azimuth']
            lag_mode: str: ['sample','ms','s']
            sampling_rate: float: sampling rate of the data to convert lags from samples to ms
        """

        
        ### Retrieve proper parameters
        
        elems_dic=self.get_dic(minlambda_select=minlambda_select)
        
        ### Select arrays 
        
        x=elems_dic['s_time']
        y=elems_dic[y_param]
        
        ### Convert raduians to degress
        if y_param=='fast':
            y=y*180/np.pi
            
        ### Change angle mode and lag_model if asked
        
        if y_param=='fast':
            if angle_mode=='azimuth':
                y=swm.trigo2azimuth(y)
        elif y_param=='lag':
            if lag_mode=='ms':
                y=y/sampling_rate*1000
            elif lag_mode=='s':
                y=y/sampling_rate
                
        ### Plot

        (ax,im,X,Y,Z,x_bins,x_diffs)=swm.plot_movehisto2d(x,y,
                x_label=x_label,y_label=y_label,title=title,ax=ax,
                **movehisto2d_kwargs)
        
        ##################################
        ### Plot time windows if asked ###
        ##################################
        
        if window_unit is not None:
            
            ### Modify seconds to proper unit
            
            if window_unit=='minute':
                x_diffs/=60
            elif window_unit=='hour':
                x_diffs/=3600
            elif window_unit=='day':
                x_diffs/=86400
            elif window_unit=='second':
                x_diffs=x_diffs
            else:
                raise ValueError('window unit must be either, day, hour, minute, second')
                
            window_label='Window size [%s]'%window_unit[0:3]
            
            ### Create new ax
            
            ax_win = ax.twinx()
            ax_win.set_yscale("log")
            
            ### Plot 
#            x_start=movehisto2d_kwargs.get('x_start',None)
#            x_end=movehisto2d_kwargs.get('x_end',None)
            
            lw_win=3
            ax_win.plot(x_bins, x_diffs, "k-",lw=lw_win,alpha=0.5)
            ax_win.plot(x_bins, x_diffs, "w-",lw=lw_win/3,alpha=0.5)
#            ax_win.set_xlim(swm.obspytime2matplotlib([x_start,x_end]))
            ### Cosmetic
            
            ax_win.set_ylim(bottom=window_bottom,top=window_top)
            ax_win.set_ylabel(window_label)

        
        ### Add vertical lines associated to start and end of eruption
        
        starteruption_time=UTCDateTime(2015,4,24,6) # Nooner and Chadwick 2016
        enderuption_time=UTCDateTime(2015,5,19)
        vline_times=[starteruption_time,enderuption_time]
        vline_times=swm.obspytime2matplotlib(vline_times)
             
        swm.plot_vlines(vline_times,ax,markercolor='w',markersize=15,
                        markeralpha=0.6,linecolor='w',linewidth=1.5)

        return ax

In [ ]:
def plot_movehisto2d(x, y,
                x_label='X', y_label='Y', title='', ax=None, vmax=None,
                **movehisto2d_kwargs):
    """
    Function made to plot an histo2d but using a moving window in both directions, this ensure better
    consistency between neighbor bins. 
    The histogram can also work when data is an obspy.UTCDateTime array, then the x_start and x_end must
    be given in UTCDateTime as well and the x_width should be given in seconds.
    UTCDatetime are converted to timestamps (seconds since 1970)
    
    Inputs
    ------
        x,y: np.array: arrays containing the data to apply histogram on (x can be UTCDateTime)
        x_width,y_width: float: width of the bins (in seconds for UTCDateTime)
        [x,y]_[start,end]: float: start and end for histogram edges
        [x,y]_over: float in [0,1]: overlap for windows [1 = full overlap]
        norm_y: Boolean: True to normalize by the maximum in each column
        smooth: Boolean: True for smoothin (Gaussian Filter)
        gaussian_[x,y]_per: float in [0,100]: width percentage for smoothing (100= filter size equal to data range)
        [x,y]_label: str
        text_list: list,str: titles to be added to the right corner of the figure
        show_counts: bool: If True it will add subplots to shown counts
        
    Ouputs
    ------
        ax_list: plt.axes: object associated to the 4 plots (top,text,mesh,right)
        X,Y,Z: meshes :
        x_bins,x_diffs : x_diffs is in seconds, whereas x_bins in matplotlib time
            x_diffs is the time resolution (i.e. the size of the bins, should be constant
            if windows is used)
    """
    
    ### Check if is made of UTCDateTimes
    
    time_flag = False
    if isinstance(x[0], UTCDateTime):
        time_flag = True
        print('X is in UTCDateTime')
        if movehisto2d_kwargs.get('x_mode', 'window') == 'window':
            print('Remember width should be given in seconds, otherwise memory error')
    
    ### Modify x_start and x_end, and x if x is time and convert to timestamps
    x_start = movehisto2d_kwargs.get('x_start', None)
    x_end = movehisto2d_kwargs.get('x_end', None)
    y_start = movehisto2d_kwargs.get('y_start', None)
    y_end = movehisto2d_kwargs.get('y_end', None)
    
    # Auto-set x_start and x_end if not provided
    if x_start is None:
        x_start = min(x) if not time_flag else min(x)
    if x_end is None:
        x_end = max(x) if not time_flag else max(x)
    
    # Default x_width if not provided (1 day in seconds for time data)
    if 'x_width' not in movehisto2d_kwargs:
        if time_flag:
            movehisto2d_kwargs['x_width'] = 86400  # 1 day in seconds
        else:
            movehisto2d_kwargs['x_width'] = (x_end - x_start) / 20  # 20 bins default
    
    if time_flag:
        if (x_start is not None) & (not isinstance(x_start, UTCDateTime)):
            raise ValueError('x_start must be given in obspy.UTCDateTime')
        if (x_end is not None) & (not isinstance(x_end, UTCDateTime)):
            raise ValueError('x_end must be given in obspy.UTCDateTime')
        x = np.array([value.timestamp for value in x])  # (seconds since 1970-01-01T00:00:00)
        x_start = x_start.timestamp if isinstance(x_start, UTCDateTime) else x_start
        x_end = x_end.timestamp if isinstance(x_end, UTCDateTime) else x_end
        movehisto2d_kwargs['x_start'] = x_start
        movehisto2d_kwargs['x_end'] = x_end

    ### Bin the data
    
    (X, Y, Z, x_bins, x_diffs) = swm.movehisto2d_bin(x, y, **movehisto2d_kwargs)
    
    ########################
    #### Start plotting ####

    #### Grid spec
    
    bottom = 0.15 if time_flag else 0.1
    
    ### Checks
    
    if ax is None:
        fig, ax = plt.subplots(gridspec_kw={'bottom': bottom, 'left': 0.15})
        
    if time_flag:
        X = np.array(swm.timestamp2matplotlib(X.ravel())).reshape(X.shape)  # transform for plotting
        x_bins = swm.timestamp2matplotlib(x_bins)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
        ax.set_xlim(swm.timestamp2matplotlib([x_start, x_end]))
    else:
        ax.set_xlim([x_start, x_end])
       
    (Xm, Ym) = swm.XY2XY_pcolormesh(X, Y)  # To ensure Pcolormesh will be centered on bins

    im = ax.pcolormesh(Xm, Ym, Z, cmap=plt.cm.get_cmap('magma'), rasterized=True, vmax=vmax)
    
    ax.set_ylim([y_start, y_end])
    
    ax.set_aspect('auto')
    if time_flag:
        ax.xaxis_date()
            
    ### Cosmetic
    
    ax.set_ylabel(y_label) 
    if not time_flag:
        ax.set_xlabel(x_label) 
   
    ###### Return
    
    return (ax, im, X, Y, Z, x_bins, x_diffs)

In [ ]:
# extract dt from results_swspy_002_tmid_1_9_2_1[eid]['result']['dt]
dts = [results_swspy_002_tmid_1_9_2_1[eid]['result']['dt'] for eid in results_swspy_002_tmid_1_9_2_1.keys()]
times = [results_swspy_002_tmid_1_9_2_1[eid]['result']['event_datetime'] for eid in results_swspy_002_tmid_1_9_2_1.keys()]

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

In [ ]:
# Convert times from UTCDateTime to matplotlib date numbers
time_nums = mdates.date2num(times)

In [ ]:
plot_movehisto2d(times, dts)

In [ ]:
def plot_movehisto2d_time_from_results(results_dict, y_param='dt',
                                       x_label='Time', y_label='Y', title='',
                                       ax=None, angle_mode='trigo', lag_mode='s',
                                       sampling_rate=200, window_unit=None,
                                       window_bottom=None, window_top=None,
                                       **movehisto2d_kwargs):
    """
    Plot 2D moving histogram of splitting parameters vs time from results dictionary.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    y_param : str
        Parameter to plot: 'dt' for delay time, 'phi' for fast direction
    x_label, y_label, title : str
        Axis labels and title
    ax : matplotlib axis
        Axis to plot on (if None, creates new figure)
    angle_mode : str
        'trigo' (default) or 'azimuth' for fast direction convention
    lag_mode : str
        'sample', 'ms', or 's' (default) for delay time units
    sampling_rate : float
        Sampling rate in Hz (default 200)
    window_unit : str
        'day', 'hour', 'minute', or 'second' for time window overlay
    window_bottom, window_top : float
        Y-axis limits for window size overlay
    **movehisto2d_kwargs : dict
        Additional arguments passed to plot_movehisto2d
        (x_width should be in seconds, y_start, y_end, y_width, etc.)
    
    Returns:
    --------
    ax : matplotlib axis
        The axis with the plot
    """
    
    # Extract times and parameter values from results
    times = []
    values = []
    
    for event_id, result_data in results_dict.items():
        result = result_data['result']
        event_time = UTCDateTime(result['event_datetime'])
        times.append(event_time)
        
        if y_param == 'dt' or y_param == 'lag':
            values.append(result['dt'])
        elif y_param == 'phi' or y_param == 'fast':
            values.append(result['phi'])
        else:
            raise ValueError(f"y_param must be 'dt', 'lag', 'phi', or 'fast', not '{y_param}'")
    
    # Convert to numpy arrays
    x = np.array(times)
    y = np.array(values)
    
    # Convert units if needed
    if y_param in ['phi', 'fast']:
        # phi is already in degrees from results
        # Change angle mode if requested
        if angle_mode == 'azimuth':
            y = swm.trigo2azimuth(y)
    elif y_param in ['dt', 'lag']:
        # Convert lag units if requested
        if lag_mode == 'sample':
            y = y * sampling_rate  # Convert seconds to samples
        elif lag_mode == 'ms':
            y = y * 1000  # Convert seconds to milliseconds
        elif lag_mode == 's':
            pass  # Already in seconds
    
    # Plot using plot_movehisto2d
    (ax, im, X, Y, Z, x_bins, x_diffs) = plot_movehisto2d(
        x, y,
        x_label=x_label, y_label=y_label, title=title, ax=ax,
        **movehisto2d_kwargs
    )
    
    ##################################
    ### Plot time windows if asked ###
    ##################################
    
    if window_unit is not None:
        
        # Convert seconds to proper unit
        if window_unit == 'minute':
            x_diffs = x_diffs / 60
        elif window_unit == 'hour':
            x_diffs = x_diffs / 3600
        elif window_unit == 'day':
            x_diffs = x_diffs / 86400
        elif window_unit == 'second':
            pass
        else:
            raise ValueError('window_unit must be either day, hour, minute, or second')
            
        window_label = f'Window size [{window_unit[:3]}]'
        
        # Create twin axis for window size
        ax_win = ax.twinx()
        ax_win.set_yscale("log")
        
        # Plot window size
        lw_win = 3
        ax_win.plot(x_bins, x_diffs, "k-", lw=lw_win, alpha=0.5)
        ax_win.plot(x_bins, x_diffs, "w-", lw=lw_win/3, alpha=0.5)
        
        # Set limits
        ax_win.set_ylim(bottom=window_bottom, top=window_top)
        ax_win.set_ylabel(window_label)
    
    ### Add vertical lines for eruption start and end
    
    starteruption_time = UTCDateTime(2015, 4, 24, 6)  # Nooner and Chadwick 2016
    enderuption_time = UTCDateTime(2015, 5, 19)
    vline_times = [starteruption_time, enderuption_time]
    vline_times = swm.obspytime2matplotlib(vline_times)
    
    swm.plot_vlines(vline_times, ax, markercolor='w', markersize=15,
                    markeralpha=0.6, linecolor='w', linewidth=1.5)
    
    return ax

In [ ]:
# Plot delay time vs time
ax = plot_movehisto2d_time_from_results(
    results_swspy_002_tmid_1_9_2_1,
    y_param='dt',
    y_label='Delay Time (s)',
    lag_mode='s',
    x_width=86400 / 12,  # 1 day in seconds
    y_start=0,
    y_end=0.15,
    window_unit='hour',
    window_bottom=0.1,
    window_top=10
)

# Plot fast direction vs time
ax = plot_movehisto2d_time_from_results(
    results_swspy_002_tmid_1_9_2_1,
    y_param='phi',
    y_label='Fast Direction (°)',
    x_width=43200 / 24,  # 12 hours in seconds
    y_start=-90,
    y_end=90
)